# Lenta: baseline трёх тематик

Education — учёба, школы, вузы и курсы. Entertainment — фильмы, музыка и игры; technology — устройства и софт. Данные отбираются воспроизводимыми правилами; это **weak labels**, не независимая человеческая разметка. Правила v3 исправляют ошибки ранних аудитов. Старые результаты сохранены в `ml/reports/history/`.

Запуск из корня: `uv sync --locked --group ml`, затем `uv run jupyter lab`. Данные и веса не коммитить.

In [ ]:
import json
import os
import sys
from pathlib import Path

cwd = Path.cwd()
ROOT = next(
    p for p in [cwd, *cwd.parents] if (p / "pyproject.toml").exists() and (p / "src/newsbot").exists()
)
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA = ROOT / "ml/data/lenta_v3"
MODELS = ROOT / "ml/models"
REPORTS = ROOT / "ml/reports"
REBUILD = False  # True: повторить подготовку и обучение
print(ROOT)

## Источник и подготовка

Lenta v1.1: 800 975 строк, SHA-256 проверяется до чтения. При отсутствии подготовленных данных следующая ячейка скачивает архив (~330 МБ). Удаляются короткие тексты, пустые идентификаторы, точные дубли и конфликты правил. Рубрики и заголовки используются только для предварительных меток, а не как признаки.

Временное разделение: train 1999–2017; validation 2018; test 2019. Просмотренные в ранних аудитах документы исключены из holdout. Близкие новости об одном событии пока не группируются.

In [ ]:
from ml.scripts.download_lenta import download
from ml.scripts.lenta_data import prepare

if REBUILD or not (DATA / "metadata.json").exists():
    archive = download(ROOT / "ml/data/lenta-ru-news.csv.bz2")
    prepare(archive, DATA)
metadata = json.loads((DATA / "metadata.json").read_text())
print(json.dumps(metadata["splits"], ensure_ascii=False, indent=2))

In [ ]:
import pandas as pd

pd.DataFrame(
    [
        {"split": name, "documents": values["rows"], "per_class": values["per_class"]}
        for name, values in metadata["splits"].items()
    ]
)

## Токенизация и обучение

Берём первые 140 слов тела статьи. Внутри Pipeline: lowercase → токенизация sklearn → TF-IDF по униграммам/биграммам → модель. TF-IDF и словарь обучаются только на Train; явные токены каждой части сохраняются в `ml/data/lenta_v3/*_tokens.jsonl`.

Сравниваются LogisticRegression и MultinomialNB. Выбор — по macro-F1 Validation, Test используется только после выбора.

In [ ]:
from ml.scripts.train_text_baseline import train

if REBUILD or not (MODELS / "text_baseline.joblib").exists():
    train(DATA, MODELS, REPORTS)
metrics = json.loads((REPORTS / "baseline_metrics.json").read_text())
assert metrics["dataset"]["splits"] == metadata["splits"]
print("Selected:", metrics["selected_model"])
print("Vocabulary:", metrics["tokenization"]["vocabulary_features"])
print(metrics["tokenization"]["splits"])

In [ ]:
pd.DataFrame(
    [
        {"model": name, "validation_macro_f1": value["macro_f1"]}
        for name, value in metrics["validation"].items()
    ]
)

In [ ]:
print("Test macro-F1 vs weak labels:", round(metrics["test"]["macro_f1"], 4))
print("Dummy macro-F1:", round(metrics["dummy_test"]["macro_f1"], 4))
pd.DataFrame(metrics["test"]["report"]).T

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    np.asarray(metrics["test"]["confusion_matrix"]), display_labels=metrics["test"]["labels"]
).plot(ax=ax, colorbar=False)
ax.set_title("2019 test; weak labels")
plt.tight_layout()
plt.show()

## Контракт для бота

Модель сохраняется вместе с токенизатором. Backend возвращает настоящий `TopicPrediction` с русской темой и confidence 0..1. Пустой текст и полностью неизвестный словарь дают «другое». Остальные сообщения вне домена модель надёжно отклонять не умеет; confidence не калиброван по человеческим меткам.

In [ ]:
from newsbot.classifier import build_classifier
from newsbot.schemas import TopicPrediction

classifier = build_classifier("tfidf", model_path=MODELS / "text_baseline.joblib")
examples = [
    "В университете начался прием абитуриентов. Для поступления нужно сдать экзамены.",
    "В кинотеатрах вышел новый фильм. Режиссер представил картину на фестивале.",
    "Компания выпустила смартфон с новым процессором и улучшенной камерой.",
    "",
]
for text in examples:
    prediction = classifier.predict(text)
    assert isinstance(prediction, TopicPrediction)
    assert 0 <= prediction.confidence <= 1
    print(text, prediction)

## TTS → Whisper → тема

Опциональный экспорт TTS использует установленный русский голос macOS Milena, PCM16 mono 16 кГц. Аудио наследует split исходного документа. По умолчанию — по 3 примера на класс в каждой части (27 файлов); `--per-class 0` озвучивает весь подготовленный корпус. Это не естественная речь и не полноценная оценка Telegram-бота.

Команды: `uv run --group ml python -m ml.scripts.make_tts_sample` и `uv run --group ml python -m ml.scripts.evaluate_tts --whisper tiny`. Whisper скачивает свои публичные веса; для работы без сети можно передать путь к локальной модели. Отчёт содержит WER и изменение классификации на тестовом аудио.

In [ ]:
tts_report = REPORTS / "tts_metrics.json"
if tts_report.exists():
    print(json.dumps(json.loads(tts_report.read_text()), ensure_ascii=False, indent=2))
else:
    print("TTS/ASR ещё не запускались в этом окружении; метрики не подставляются.")

## Ограничения

- Рубрики и правила могут ошибаться, особенно у политических и бытовых сюжетов. Высокое согласие с weak labels не доказывает семантическую точность.
- Классы отбираются и балансируются; это не реальное распределение пользовательских сообщений.
- Технологии представлены в основном устройствами и софтом, образование — формальным обучением; другие подтемы могут распознаваться хуже.
- Нет независимого человеческого gold-набора и проверки на реальных голосовых.
- Полные условия использования исходных текстов Lenta требуют уточнения: публичная доступность не является лицензией.

Подробности, данные для передачи и следующие этапы — в `ml/reports/DATA_HANDOFF.md`.